# Detecting Cannabis–Alcohol Substitution in Aggregate Sales Data
## Statistical Power and the Limits of Small-Sample Synthetic Control

This notebook is the presentation layer of a hybrid pipeline:

1. **Data prep (Python, below)** — parses the NIAAA surveillance file and
   writes tidy panels to `Data/processed/`.
2. **Estimation (R)** — `run_all.R` runs the estimator horse race
   (partially-pooled staggered ASCM via `augsynth::multisynth` [primary],
   `synthdid`, `gsynth` IFE + matrix completion, Callaway–Sant'Anna via `did`,
   TWFE), pre-specified sample rules, conformal prediction intervals (`scpi`),
   in-time placebos, pooled randomization inference, the design-based power
   simulation, and the robustness suite. Every step caches to `Results/cache/`
   and exports tidy CSVs to `Results/csv/`.
3. **Figures + tables (Python, below)** — every figure and number is generated
   programmatically from `Results/csv/` (no hand-typed statistics).

Cold-run the full pipeline by deleting `Results/cache/` first.


## 1. Data preparation (NIAAA panel → tidy CSVs)

In [ ]:
# Data preparation: NIAAA panel -> tidy CSVs consumed by the R estimation
# pipeline. Ported from the v1 notebook loader (archive_v1/).
import os
import numpy as np
import pandas as pd

DATA_PATH = 'Data/pcyr1970-2023.txt'
OUT_DIR = 'Data/processed'
YEAR_START, YEAR_END = 2000, 2023

col_names = ['year', 'fips', 'beverage', 'gal_bev', 'gal_eth',
             'pop14', 'pc_eth14', 'decile14', 'pop21', 'pc_eth21',
             'decile21', 'data_source', 'abv', 'eth_tvabv']

# Recreational retail opening years (treatment = first full retail year).
treatment_years = {
    8: 2014, 53: 2014, 41: 2015, 2: 2016, 32: 2017, 6: 2018, 25: 2018,
    26: 2019, 17: 2020, 23: 2020, 4: 2021, 30: 2022, 34: 2022, 35: 2022,
    36: 2022, 50: 2022, 44: 2022, 9: 2023, 24: 2023, 29: 2023,
}
PRIMARY_STATES = [8, 53, 41, 2, 32, 6, 25]   # the 7 analyzed treated states
# Any state with recreational legalization through 2023 (incl. DC) — excluded
# from the never-treated donor pool.
rec_legal_fips = {2, 4, 6, 8, 9, 11, 17, 23, 24, 25, 26, 29, 30, 32, 34, 35,
                  36, 41, 44, 50, 53}

fips_to_state = {
    1: 'Alabama', 2: 'Alaska', 4: 'Arizona', 5: 'Arkansas', 6: 'California',
    8: 'Colorado', 9: 'Connecticut', 10: 'Delaware', 11: 'DC', 12: 'Florida',
    13: 'Georgia', 15: 'Hawaii', 16: 'Idaho', 17: 'Illinois', 18: 'Indiana',
    19: 'Iowa', 20: 'Kansas', 21: 'Kentucky', 22: 'Louisiana', 23: 'Maine',
    24: 'Maryland', 25: 'Massachusetts', 26: 'Michigan', 27: 'Minnesota',
    28: 'Mississippi', 29: 'Missouri', 30: 'Montana', 31: 'Nebraska',
    32: 'Nevada', 33: 'New Hampshire', 34: 'New Jersey', 35: 'New Mexico',
    36: 'New York', 37: 'North Carolina', 38: 'North Dakota', 39: 'Ohio',
    40: 'Oklahoma', 41: 'Oregon', 42: 'Pennsylvania', 44: 'Rhode Island',
    45: 'South Carolina', 46: 'South Dakota', 47: 'Tennessee', 48: 'Texas',
    49: 'Utah', 50: 'Vermont', 51: 'Virginia', 53: 'Washington',
    54: 'West Virginia', 55: 'Wisconsin', 56: 'Wyoming'
}


def build_processed_data():
    os.makedirs(OUT_DIR, exist_ok=True)
    raw = pd.read_csv(DATA_PATH, skiprows=129, sep=r'\s+', header=None,
                      names=col_names, na_values=['.'])
    raw['pc_eth21'] = raw['pc_eth21'] / 10_000
    raw['pc_eth14'] = raw['pc_eth14'] / 10_000
    raw = raw[raw['fips'].between(1, 56) & (raw['fips'] != 11)].copy()  # drop DC

    bev_map = {1: 'spirits', 2: 'wine', 3: 'beer', 4: 'total'}
    df = raw[raw['year'].between(YEAR_START, YEAR_END)].copy()
    df['beverage'] = df['beverage'].map(bev_map)
    df = df[df['beverage'].notna() & df['pc_eth21'].notna() & (df['pc_eth21'] > 0)]
    df['state'] = df['fips'].map(fips_to_state)
    df['log_pc_eth21'] = np.log(df['pc_eth21'])
    # 21+ population (same for all beverage rows of a state-year); used by the
    # first-stage per-capita cannabis sales figure.
    pop21 = (df[df['beverage'] == 'total'][['year', 'fips', 'pop21']]
             .drop_duplicates())

    # Beer+wine combined ethanol (spirits-excluded outcome for the WA check):
    bw = (df[df['beverage'].isin(['beer', 'wine'])]
          .groupby(['year', 'fips', 'state'], as_index=False)['pc_eth21'].sum())
    bw['beverage'] = 'beer_wine'
    bw['log_pc_eth21'] = np.log(bw['pc_eth21'])
    panel = pd.concat([df[['year', 'fips', 'state', 'beverage', 'pc_eth21',
                           'log_pc_eth21']], bw], ignore_index=True)

    # Keep only states with a balanced total-ethanol series 2000-2023
    tot = panel[panel['beverage'] == 'total']
    counts = tot.groupby('fips')['year'].nunique()
    balanced = counts[counts == (YEAR_END - YEAR_START + 1)].index
    panel = panel[panel['fips'].isin(balanced)].copy()
    panel.sort_values(['beverage', 'fips', 'year']).to_csv(
        f'{OUT_DIR}/panel_long.csv', index=False)
    pop21[pop21['fips'].isin(balanced)].sort_values(['fips', 'year']).to_csv(
        f'{OUT_DIR}/population_21.csv', index=False)

    treat = pd.DataFrame({'fips': sorted(fips_to_state)})
    treat = treat[treat['fips'] != 11]
    treat['state'] = treat['fips'].map(fips_to_state)
    treat['t0'] = treat['fips'].map(treatment_years).astype('Int64')
    treat['primary_treated'] = treat['fips'].isin(PRIMARY_STATES).astype(int)
    treat['ever_rec_2023'] = treat['fips'].isin(rec_legal_fips).astype(int)
    treat.to_csv(f'{OUT_DIR}/treatment.csv', index=False)

    n_states = panel[panel['beverage'] == 'total']['fips'].nunique()
    n_donor = int(((treat['ever_rec_2023'] == 0)
                   & treat['fips'].isin(balanced)).sum())
    print(f"panel_long.csv: {n_states} states x {YEAR_END-YEAR_START+1} years, "
          f"{panel['beverage'].nunique()} beverage series")
    print(f"treatment.csv: {int(treat['primary_treated'].sum())} primary treated, "
          f"{n_donor} never-treated donors")
    return panel, treat

## 2. R estimation pipeline (cached)

In [ ]:
# Run the cached R estimation pipeline (no-op when caches are warm).
import subprocess, time

RSCRIPT = r"C:\Program Files\R\R-4.4.1\bin\Rscript.exe"
t0 = time.time()
proc = subprocess.run([RSCRIPT, "run_all.R"], capture_output=True, text=True)
print(proc.stdout[-4000:] if len(proc.stdout) > 4000 else proc.stdout)
print(proc.stderr[-2500:])
assert proc.returncode == 0, "R pipeline failed — see stderr above"
print(f"R pipeline wall time this run: {(time.time() - t0) / 60:.1f} min")

import pandas as pd
print(pd.read_csv("Results/runtimes.csv").to_string(index=False))


## 3. Port-faithfulness check
The v1 hand-coded SCM solver and R `augsynth` must agree on the identical spec before any redesigned number is trusted.

In [ ]:
# Port-faithfulness check: the v1 hand-coded SCM solver (scipy SLSQP) and R
# augsynth (progfunc="None", scm=TRUE) should agree closely on the identical
# spec (outcome-only, never-treated donors, 2000-2019). Run after the R
# pipeline has written Results/csv/state_scm_att.csv.
import numpy as np
import pandas as pd
from scipy.optimize import minimize


def solve_scm_v1(Y_treated_pre, Y_donors_pre):
    J = Y_donors_pre.shape[1]

    def objective(w):
        gap = Y_treated_pre - Y_donors_pre @ w
        return gap @ gap

    result = minimize(objective, x0=np.ones(J) / J, method='SLSQP',
                      bounds=[(0, 1)] * J,
                      constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1.0},
                      options={'maxiter': 1000, 'ftol': 1e-12})
    w = result.x.copy()
    w[w < 1e-4] = 0
    if w.sum() > 0:
        w /= w.sum()
    return w


def sanity_anchor():
    panel = pd.read_csv('Data/processed/panel_long.csv')
    treat = pd.read_csv('Data/processed/treatment.csv')
    never = treat[treat['ever_rec_2023'] == 0]['fips'].tolist()
    tot = panel[(panel['beverage'] == 'total') & panel['year'].between(2000, 2019)]
    wide = tot.pivot_table(index='year', columns='fips', values='log_pc_eth21')

    r_att = pd.read_csv('Results/csv/state_scm_att.csv')
    rows = []
    for f, t0 in [(8, 2014), (6, 2018)]:
        pre = wide.index < t0
        post = ~pre
        Yt = wide[f].values
        Yd = wide[never].values
        w = solve_scm_v1(Yt[pre], Yd[pre])
        gap = Yt - Yd @ w
        att_py = gap[post].mean()
        att_r = r_att[(r_att['fips'] == f) &
                      (r_att['estimator'] == 'classic_scm')]['att'].iloc[0]
        rows.append({'fips': f, 'att_python_v1': att_py, 'att_r_augsynth': att_r,
                     'abs_diff': abs(att_py - att_r)})
    out = pd.DataFrame(rows)
    assert (out['abs_diff'] < 0.005).all(), \
        f"Port check failed — solvers disagree by >0.5 log points:\n{out}"
    print("Port-faithfulness check PASSED (max |diff| = "
          f"{out['abs_diff'].max():.5f} log points)")
    return out

print(sanity_anchor())

## 4. Covariate robustness (Kaul et al. 2022 lag-subset spec)

In [ ]:
# Covariate robustness (Kaul, Klossner, Pfeifer & Schieler 2022 JAE): matching
# on all outcome lags renders covariates irrelevant, so the primary spec is
# outcome-only by design. This robustness spec matches on a lag SUBSET
# (odd pre-period years) plus covariates measured PRE-T0 per state (mean of
# the three years before T0), and compares the classic-SCM ATT to the
# outcome-only ATT for each clean-fit state.
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize

COV_FILES = {
    'real_gdp_pc': 'Data/external/bea_gdp_pc.csv',
    'unemp_rate': 'Data/external/bls_unemployment.csv',
    'share_20_34': 'Data/external/age_share_20_34.csv',
    'beer_tax_per_gal': 'Data/external/beer_tax.csv',
}


def _solve_scm(Z_treated, Z_donors):
    J = Z_donors.shape[1]

    def objective(w):
        gap = Z_treated - Z_donors @ w
        return gap @ gap

    res = minimize(objective, x0=np.ones(J) / J, method='SLSQP',
                   bounds=[(0, 1)] * J,
                   constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1.0},
                   options={'maxiter': 1000, 'ftol': 1e-12})
    w = res.x.copy()
    w[w < 1e-4] = 0
    if w.sum() > 0:
        w /= w.sum()
    return w


def _load_covariate_panels():
    panels = {}
    for var, path in COV_FILES.items():
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        val_col = [c for c in df.columns if c not in
                   ('fips', 'state', 'year', 'source_url')][0]
        panels[var] = df[['fips', 'year', val_col]].rename(columns={val_col: var})
    return panels


def covariate_lag_subset_spec():
    panels = _load_covariate_panels()
    if not panels:
        print('No yearly covariate panels available — spec skipped.')
        return None
    panel = pd.read_csv('Data/processed/panel_long.csv')
    treat = pd.read_csv('Data/processed/treatment.csv')
    rules = pd.read_csv('Results/csv/sample_rules.csv')
    never = treat[treat['ever_rec_2023'] == 0]['fips'].tolist()
    tot = panel[(panel['beverage'] == 'total') & panel['year'].between(2000, 2019)]
    wide = tot.pivot_table(index='year', columns='fips', values='log_pc_eth21')

    rows = []
    for _, r in rules[rules['included'] == 1].iterrows():
        f, t0 = int(r['fips']), int(r['t0'])
        # Covariates measured pre-T0: mean over [t0-3, t0-1], standardized
        # across the treated state + donors.
        cov = {}
        used = []
        for var, p in panels.items():
            m = (p[p['year'].between(t0 - 3, t0 - 1)]
                 .groupby('fips')[var].mean())
            if f in m.index and all(d in m.index for d in never):
                cov[var] = m
                used.append(var)
        pre_years = [y for y in wide.index if y < t0]
        odd_lags = [y for y in pre_years if y % 2 == 1]
        post = wide.index >= t0

        # Every predictor row (each odd-year lag AND each covariate) is
        # z-scored across donors, so all predictors enter the match at equal
        # scale. Without this, standardized covariates carry ~20x the weight
        # of raw log-point lags and the fit degenerates.
        def stack_raw(unit):
            lags = wide.loc[odd_lags, unit].values
            cv = np.array([cov[v][unit] for v in used])
            return np.concatenate([lags, cv])

        Zd_raw = np.column_stack([stack_raw(d) for d in never])
        mu = Zd_raw.mean(axis=1)
        sd = Zd_raw.std(axis=1)
        Zt = (stack_raw(f) - mu) / sd
        Zd = (Zd_raw - mu[:, None]) / sd[:, None]
        w = _solve_scm(Zt, Zd)
        gap = wide[f].values - wide[never].values @ w
        att_cov = gap[post].mean()

        # outcome-only all-lags benchmark (primary-style classic SCM)
        w0 = _solve_scm(wide.loc[pre_years, f].values,
                        wide.loc[pre_years, never].values)
        gap0 = wide[f].values - wide[never].values @ w0
        att0 = gap0[post].mean()
        rows.append({'fips': f, 'state': r['state'], 't0': t0,
                     'att_outcome_only': att0, 'att_lagsubset_cov': att_cov,
                     'covariates_used': '+'.join(used)})
    out = pd.DataFrame(rows)
    out.to_csv('Results/csv/covariate_lag_subset.csv', index=False)
    print(out.to_string(index=False))
    return out

covariate_lag_subset_spec()

## 5. Figures (all from `Results/csv/`)

In [ ]:
# Figures for the redesigned paper — every figure built from Results/csv/*.csv
# (no hand-typed numbers). Style carried over from the v1 notebook.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (10, 6), 'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 12, 'legend.fontsize': 10,
    'figure.dpi': 150, 'axes.spines.top': False, 'axes.spines.right': False,
})
COLORS = {'treated': '#1f77b4', 'synth_scm': '#d62728', 'synth_ascm': '#2ca02c',
          'placebo': '#cccccc', 'donor_avg': '#999999', 'accent': '#ff7f0e'}
CSV = 'Results/csv'
FIG = 'Results/figures'
os.makedirs(FIG, exist_ok=True)

EST_LABELS = {
    'multisynth': 'Partially-pooled ASCM\n(multisynth, jackknife)',
    'sdid': 'Synthetic DiD\n(placebo SE)',
    'gsynth_ife': 'Gen. synthetic control\n(IFE, param. bootstrap)',
    'matrix_completion': 'Matrix completion\n(bootstrap)',
    'callaway_santanna': "Callaway-Sant'Anna\n(multiplier bootstrap)",
    'twfe': 'TWFE\n(clustered SE)',
}
EST_ORDER = ['multisynth', 'sdid', 'gsynth_ife', 'matrix_completion',
             'callaway_santanna', 'twfe']


def load_estimator_summary():
    """One row per pooled estimator: att, se, ci."""
    rows = []
    ovj = pd.read_csv(f'{CSV}/multisynth_overall_jack.csv').iloc[0]
    rows.append(('multisynth', ovj['att'], ovj['se_jack']))
    sd = pd.read_csv(f'{CSV}/sdid_primary.csv')
    agg = sd[sd['states'] == 'Aggregate'].iloc[0]
    rows.append(('sdid', agg['tau'], agg['se']))
    gs = pd.read_csv(f'{CSV}/gsynth_primary.csv')
    for _, r in gs.iterrows():
        rows.append((r['estimator'], r['att'], r['se']))
    ct = pd.read_csv(f'{CSV}/cs_twfe_primary.csv')
    for _, r in ct.iterrows():
        rows.append((r['estimator'], r['att'], r['se']))
    df = pd.DataFrame(rows, columns=['estimator', 'att', 'se'])
    df['ci_lo'] = df['att'] - 1.96 * df['se']
    df['ci_hi'] = df['att'] + 1.96 * df['se']
    df['estimator'] = pd.Categorical(df['estimator'], EST_ORDER, ordered=True)
    return df.sort_values('estimator').reset_index(drop=True)


def fig_forest_estimators():
    """Headline figure: all pooled estimators on one axis + RI p-value."""
    df = load_estimator_summary()
    ri = pd.read_csv(f'{CSV}/ri_pooled.csv').iloc[0]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    y = np.arange(len(df))[::-1]
    ax.errorbar(df['att'], y, xerr=1.96 * df['se'], fmt='o', color=COLORS['treated'],
                ecolor=COLORS['treated'], elinewidth=2, capsize=4, markersize=7)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([EST_LABELS[e] for e in df['estimator']], fontsize=9)
    ax.set_xlabel('Pooled ATT on log per-capita ethanol (95% CI)')
    ax.set_title('Pooled Effect of Recreational Legalization: Six Estimators\n'
                 '(clean-fit states, never-treated donors, 2000–2019)')
    for yi, (_, r) in zip(y, df.iterrows()):
        ax.text(r['ci_hi'] + 0.004, yi, f"{np.exp(r['att'])-1:+.1%}",
                va='center', fontsize=9)
    # Widen the right margin so the joint-null annotation sits in empty space
    # to the right of the estimate labels rather than over the TWFE label.
    x0, x1 = ax.get_xlim()
    ax.set_xlim(x0, x1 + 0.08)
    ax.text(0.985, 0.04,
            f"Joint null test (randomization inference,\n"
            f"N={int(ri['n_draws'])} draws): p = {ri['p_two_sided']:.2f}",
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round', fc='white', ec='#cccccc'))
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n1_forest_estimators.pdf', bbox_inches='tight')
    plt.show()
    return df


def fig_bias_bands():
    """Systematic in-time placebos: fake-ATT distributions vs real estimates."""
    sp = pd.read_csv(f'{CSV}/intime_placebo_single.csv').dropna(subset=['fake_att'])
    real = pd.read_csv(f'{CSV}/state_scm_att.csv')
    ests = ['classic_scm', 'ridge_ascm', 'sdid', 'gsynth_ife']
    labels = {'classic_scm': 'Classic SCM', 'ridge_ascm': 'Ridge ASCM',
              'sdid': 'Synthetic DiD', 'gsynth_ife': 'gsynth (IFE)'}
    clean = ['Colorado', 'Washington', 'Oregon', 'California', 'Massachusetts']
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, e in enumerate(ests):
        vals = sp[sp['estimator'] == e]['fake_att'].values
        x = rng.normal(i, 0.055, len(vals))
        ax.scatter(x, vals, s=15, color=COLORS['placebo'], edgecolor='#9a9a9a',
                   linewidth=0.4, alpha=0.7, zorder=2)
        m, sd = vals.mean(), vals.std()
        ax.hlines(m, i - 0.3, i + 0.3, color='#444444', lw=2.2, zorder=3)
        ax.hlines([m - 2 * sd, m + 2 * sd], i - 0.22, i + 0.22,
                  color='#444444', lw=1, ls=(0, (4, 3)), zorder=3)
    # real single-state estimates, spread within the column so they do not overlap
    for i, e in enumerate(['classic_scm', 'ridge_ascm']):
        rr = real[(real['estimator'] == e) & real['state'].isin(clean)]
        rr = rr.sort_values('state')
        offs = np.linspace(-0.17, 0.17, len(rr)) if len(rr) > 1 else np.zeros(len(rr))
        ax.scatter(i + offs, rr['att'], s=46, marker='D', color=COLORS['treated'],
                   edgecolor='white', linewidth=0.8, zorder=5,
                   label='Real estimates (clean-fit states)' if i == 0 else None)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(range(len(ests)))
    ax.set_xticklabels([labels[e] for e in ests])
    ax.set_xlim(-0.5, len(ests) - 0.5)
    ax.margins(y=0.13)
    ax.set_ylabel('Fake ATT (log points)')
    ax.set_title('Empirical Bias Bands: Backdated Placebos at Every Feasible Fake $T_0$\n'
                 '(grey = fake ATTs, pre-real-treatment data only; bars = mean ± 2 SD; '
                 'blue = real estimates)')
    ax.legend(frameon=False, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n2_bias_bands.pdf', bbox_inches='tight')
    plt.show()


def fig_fake2009():
    """Pooled fake-2009 diagnostic across all estimators."""
    df = pd.read_csv(f'{CSV}/intime_placebo_pooled2009.csv')
    ipj = pd.read_csv(f'{CSV}/intime_pooled_jack.csv').set_index('fake_t0')
    df.loc[df['estimator'] == 'multisynth', 'se'] = float(ipj.loc[2009, 'se_jack'])
    df['estimator'] = pd.Categorical(df['estimator'], EST_ORDER, ordered=True)
    df = df.sort_values('estimator').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    y = np.arange(len(df))[::-1]
    ax.errorbar(df['att'], y, xerr=1.96 * df['se'], fmt='s',
                color=COLORS['synth_scm'], elinewidth=2, capsize=4, markersize=7)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([EST_LABELS[e] for e in df['estimator']], fontsize=9)
    ax.set_xlabel('Fake pooled ATT, T0 = 2009, panel ends 2013 (95% CI)')
    ax.set_title('In-Time Placebo, Pooled: Does Any Estimator Absorb the Western Drift?')
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n3_fake2009_pooled.pdf', bbox_inches='tight')
    plt.show()


def fig_power():
    """Calibrated power curves for all five estimators, plus the primary\n    estimator's jackknife curve (the inference the paper reports)."""
    pt = pd.read_csv(f'{CSV}/power_results.csv')
    mde = pd.read_csv(f'{CSV}/power_mde.csv').set_index('rule')
    ms = pt[pt['estimator'] == 'multisynth'].sort_values('delta')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(ms['delta'] * 100, ms['reject_ri'], '-o', color=COLORS['treated'],
            lw=2.2, markersize=7, label='multisynth, RI-calibrated test (size 5%)')
    r3 = pd.read_csv(f'{CSV}/power_results_3rules.csv')
    ms3 = r3[r3['estimator'] == 'multisynth'].sort_values('delta')
    ax.plot(ms3['delta'] * 100, ms3['reject_jack'], '-^', color='#1d5a55',
            lw=2, markersize=6, label='multisynth, jackknife t-test')
    markers = {'sdid': '^', 'gsynth_ife': 's', 'matrix_completion': 'D',
               'callaway_santanna': 'v'}
    cols = {'sdid': '#d62728', 'gsynth_ife': '#2ca02c',
            'matrix_completion': '#9467bd', 'callaway_santanna': '#ff7f0e'}
    for e in markers:
        sub = pt[pt['estimator'] == e].sort_values('delta')
        ax.plot(sub['delta'] * 100, sub['reject_ri'], '-', lw=1.2, alpha=0.85,
                marker=markers[e], markersize=5, color=cols[e],
                label=EST_LABELS[e].split('\n')[0] + ' (RI rule)')
    ax.axhline(0.8, color='gray', ls='--', lw=0.8)
    ax.axhline(0.05, color='gray', ls=':', lw=0.8)
    ax.text(-0.6, 0.815, '80% power', fontsize=9, color='gray')
    ax.text(-0.6, 0.065, '5% nominal size', fontsize=9, color='gray')
    mde3 = pd.read_csv(f'{CSV}/power_mde_3rules.csv').set_index('rule')
    mj = mde3.loc['jackknife']
    if not np.isnan(mj['mde_delta']):
        ax.axvline(mj['mde_delta'] * 100, color='#1d5a55', ls=':', lw=1)
        ax.text(mj['mde_delta'] * 100 + 0.3, 0.62,
                f"jackknife MDE \u2248 {mj['mde_delta']:.1%}",
                fontsize=9, color='#1d5a55', ha='left')
    mri = mde.loc['ri_calibrated']
    if not np.isnan(mri['mde_delta']):
        ax.axvline(mri['mde_delta'] * 100, color=COLORS['treated'], ls=':', lw=1)
        ax.text(mri['mde_delta'] * 100 - 0.3, 0.45,
                f"primary MDE ≈ {mri['mde_delta']:.1%}\n"
                f"(≈{mri['mde_drinks_per_month']:.1f} drinks/person/month)",
                fontsize=9, color=COLORS['treated'], ha='right')
    ax.set(xlabel='True injected effect δ (percent of per-capita ethanol)',
           ylabel='Rejection rate at 5% level', ylim=(-0.04, 1.06),
           title='Design-Based Power: Staggered Pseudo-Treatment in Never-Treated States')
    ax.legend(frameon=False, fontsize=8.5, loc='lower left',
              bbox_to_anchor=(0.01, 0.07))
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n4_power_curve.pdf', bbox_inches='tight')
    plt.show()


def fig_state_gaps():
    """Descriptive per-state gap paths, classic vs ridge ASCM, 7 states."""
    gaps = pd.read_csv(f'{CSV}/state_scm_gaps.csv')
    sr = pd.read_csv(f'{CSV}/sample_rules.csv')
    t0s = dict(zip(sr['state'], sr['t0']))
    incl = dict(zip(sr['state'], sr['included']))
    states = ['Colorado', 'Washington', 'Oregon', 'Alaska', 'Nevada',
              'California', 'Massachusetts']
    fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharex=True)
    for ax, st in zip(axes.flat, states):
        for est, c, ls in [('classic_scm', COLORS['synth_scm'], '-'),
                           ('ridge_ascm', COLORS['synth_ascm'], ':')]:
            g = gaps[(gaps['state'] == st) & (gaps['estimator'] == est)]
            ax.plot(g['year'], g['gap'], ls, color=c, lw=2,
                    label=est.replace('_', ' '))
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(t0s[st], color='gray', ls='--', lw=0.8, alpha=0.6)
        flag = '' if incl[st] else '  [excluded by fit rule]'
        ax.set_title(f"{st} (T0={t0s[st]}){flag}",
                     color='black' if incl[st] else '#999999')
        ax.legend(frameon=False, fontsize=7)
    axes.flat[-1].set_visible(False)
    fig.suptitle('State-by-State SCM Gaps (descriptive; 2000–2019, '
                 'never-treated donors)', fontsize=14, y=1.0)
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n5_state_gaps.pdf', bbox_inches='tight')
    plt.show()


def fig_scpi():
    """Conformal prediction intervals for the clean-fit states."""
    sc = pd.read_csv(f'{CSV}/scpi_conformal.csv')
    states = sc['state'].unique()
    fig, axes = plt.subplots(1, len(states), figsize=(4 * len(states), 4.2),
                             sharey=True)
    for ax, st in zip(np.atleast_1d(axes), states):
        g = sc[sc['state'] == st]
        ax.fill_between(g['year'], g['gap_lo'], g['gap_hi'],
                        color=COLORS['placebo'], alpha=0.7,
                        label='90% conformal PI')
        ax.plot(g['year'], g['gap'], '-o', color=COLORS['treated'], lw=2,
                markersize=4)
        ax.axhline(0, color='black', lw=0.8)
        ax.set_title(st)
        ax.set_xticks(g['year'][::2])
    np.atleast_1d(axes)[0].set_ylabel('Post-period gap (log points)')
    np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
    fig.suptitle('Conformal Prediction Intervals (Chernozhukov-Wüthrich-Zhu / scpi)',
                 fontsize=13, y=1.03)
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n6_scpi_conformal.pdf', bbox_inches='tight')
    plt.show()


def fig_cs_eventstudy():
    dyn = pd.read_csv(f'{CSV}/cs_eventstudy.csv')
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for pre, color, label in [(True, 'gray', 'Pre-treatment'),
                              (False, COLORS['treated'], 'Post-treatment')]:
        g = dyn[dyn['event_time'] < 0] if pre else dyn[dyn['event_time'] >= 0]
        ax.errorbar(g['event_time'], g['att'],
                    yerr=g['crit_val'] * g['se'], fmt='o', color=color,
                    markersize=5, capsize=3, lw=1.2, label=label)
    ax.axhline(0, color='black', lw=0.7)
    ax.axvline(-0.5, color='gray', ls='--', lw=0.8, alpha=0.6)
    ax.set(xlabel='Years relative to retail opening',
           ylabel='Effect on log per-capita ethanol',
           title="Callaway-Sant'Anna Event Study (primary sample)")
    ax.legend(frameon=False, loc='upper left', fontsize=9,
              title='95% simultaneous CIs', title_fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n7_cs_eventstudy.pdf', bbox_inches='tight')
    plt.show()


def fig_first_stage():
    """Cannabis sales per adult (21+) ramp + dose vs. state ATT scatter."""
    path = 'Data/external/cannabis_sales_annual.csv'
    if not os.path.exists(path):
        print('first-stage data not available — figure skipped')
        return None
    cs = pd.read_csv(path)
    pop = pd.read_csv('Data/processed/population_21.csv')
    cs = cs.merge(pop, on=['fips', 'year'], how='left')
    # 2024 fiscal-year rows have no NIAAA population — drop from the figure
    cs = cs.dropna(subset=['pop21'])
    cs['sales_pc'] = cs['sales_usd'] / cs['pop21']
    sr = pd.read_csv(f'{CSV}/sample_rules.csv')
    t0s = dict(zip(sr['fips'], sr['t0']))
    st_att = pd.read_csv(f'{CSV}/state_scm_att.csv')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
    for st, g in cs.groupby('state'):
        g = g.sort_values('year')
        basis = g['year_basis'].iloc[0]
        ax1.plot(g['year'], g['sales_pc'], '-o', lw=2, markersize=4,
                 label=f"{st}{' (FY)' if basis == 'FY' else ''}")
    ax1.set(xlabel='Year', ylabel='Legal cannabis sales per adult 21+ ($, nominal)',
            title='First Stage: Per-Adult Legal Cannabis Sales')
    ax1.legend(frameon=False, fontsize=9)

    # Dose = mean per-adult sales in the first two post years; ATT from the
    # descriptive classic SCM.
    rows = []
    for f, g in cs.groupby('fips'):
        t0 = t0s.get(f)
        if t0 is None:
            continue
        dose = g[g['year'].between(t0, t0 + 1)]['sales_pc'].mean()
        att = st_att[(st_att['fips'] == f) &
                     (st_att['estimator'] == 'classic_scm')]['att']
        if len(att) and pd.notna(dose):
            rows.append({'state': g['state'].iloc[0], 'dose': dose,
                         'att': att.iloc[0]})
    dd = pd.DataFrame(rows)
    ax2.scatter(dd['dose'], dd['att'], s=70, color=COLORS['treated'], zorder=3)
    for _, r in dd.iterrows():
        ax2.annotate(r['state'], (r['dose'], r['att']),
                     textcoords='offset points', xytext=(6, 4), fontsize=9)
    ax2.axhline(0, color='black', lw=0.8)
    ax2.set(xlabel='Mean per-adult sales, first two post years ($)',
            ylabel='Classic SCM ATT (log points)',
            title='Dose vs. Estimated Effect (descriptive)')
    plt.tight_layout()
    plt.savefig(f'{FIG}/fig_n8_first_stage.pdf', bbox_inches='tight')
    plt.show()
    return dd


def all_figures():
    df = fig_forest_estimators()
    fig_bias_bands()
    fig_fake2009()
    try:
        fig_power()
    except FileNotFoundError:
        print('power results not ready — fig_n4 skipped')
    fig_state_gaps()
    fig_scpi()
    fig_cs_eventstudy()
    fig_first_stage()
    return df


estimator_summary = all_figures()

## 6. Paper table fragments
Every table row and in-text statistic in the paper is generated here (`paper/gen/*.tex`) from `Results/csv/`. No number in the manuscript is hand-typed.

In [ ]:
# Generates every LaTeX table fragment and in-text number macro for the
# paper via notebook_src/make_tables.py, extracted
# from this cell so the generator is a reviewable script. No statistic in
# either paper is hand-typed.
import subprocess
import sys

subprocess.run([sys.executable, 'notebook_src/make_tables.py'], check=True)
